# *Intro*

In [ ]:
import numpy as np
import pandas as pd

from collections import Counter

import copy
from copy import deepcopy

import spacy
nlp = spacy.load('en_core_web_sm')

import string

import matplotlib.pyplot as plt

import re

import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Dense, Embedding, Input, Bidirectional

import nltk
from nltk.corpus import stopwords
from nltk import ngrams
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
from nltk.tokenize import TweetTokenizer
from nltk.tokenize import word_tokenize

!pip install ekphrasis -U
import ekphrasis
from ekphrasis.classes.segmenter import Segmenter
from ekphrasis.classes.spellcorrect import SpellCorrector
sp = SpellCorrector(corpus="twitter")
seg = Segmenter(corpus="twitter")

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, TfidfTransformer
from sklearn.utils import class_weight
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')

!pip install emoji_translate
from emoji_translate.emoji_translate import Translator

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/hate-train.csv')

validation = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/hate-validation.csv')

test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/hate-test.csv')

In [ ]:
train.head()

## Some descriptive statistics

In [ ]:
train['gold_label'].value_counts()

In [ ]:
count_dict = train['gold_label'].value_counts().to_dict()

perc_dict = {key: (round(count_dict[key]/sum(count_dict.values()), 3)) for key in count_dict}

In [ ]:
labels = {0: 'Gender', 1: 'Race', 2: 'Sexuality',
 3: 'Relgion', 4: 'Origin', 5: 'Disability',
 6: 'Age', 7: 'Not hateful'}

In [ ]:
count_dict1 = {labels[key]: count_dict[key] for key in count_dict}

perc_dict1 = {labels[key]: perc_dict[key] for key in perc_dict}

In [ ]:
# create 2 graphs
fig, axs = plt.subplots(2, figsize=(5,9))

# create bar graphs of label counts
bar_container0 = axs[0].bar(count_dict1.keys(), count_dict1.values())
axs[0].bar_label(bar_container0, label_type='edge')
axs[0].set_xlabel('Hate Speech Type')
axs[0].set_ylabel('Count')
axs[0].set_title('Count by Hate Speech Type in the Training Data')
axs[0].tick_params(axis='x', labelrotation=30)

# create bar graphs of label percentages
bar_container1 = axs[1].bar(perc_dict1.keys(), perc_dict1.values())
axs[1].bar_label(bar_container1, label_type='edge')
axs[1].set_xlabel('Hate Speech Type')
axs[1].set_ylabel('Percentage (%)')
axs[1].set_title('Percentage (%) by Hate Speech Type in the Training Data')
axs[1].tick_params(axis='x', labelrotation=30)

plt.tight_layout()
plt.show()

In [ ]:
train['string length'] = train['text'].apply(lambda words: len(words.split(" ")))
mean_length = np.round(train['string length'].mean()).astype(int)

train['string length'].plot.hist(label='_nolegend_')
plt.axvline(x=mean_length, color='k', linestyle='--', label=f'Mean length (={mean_length})');
plt.xlabel('Length of text')
plt.title('Distribution of Text Lengths in the Training Data')
plt.legend()

plt.show()

## A look into the data

In [ ]:
#top 10 words of each class
from collections import Counter

df = train.copy()
df['text'] = df['text'].apply(lambda x: x.split())
df['text'] = df['text'].apply(lambda sent: [word.lower() for word in sent])
df['text'] = df['text'].apply(lambda sent: [word for word in sent if word not in ["@user","{url}"]])

stop_words = set(stopwords.words('english'))
df['text'] = df['text'].apply(lambda sent: [word for word in sent if word not in stop_words])

import string
df['text'] = df['text'].apply(lambda sent: [word for word in sent if word not in string.punctuation])


for i in range(8):
    #
    filtered_df = df[df['gold_label'] == i]

    #
    flat_data_i = [item for sublist in filtered_df['text'] for item in sublist]

    #
    top_10_words = Counter(flat_data_i).most_common(10)

    print(f'Class {i}:')
    print(top_10_words)

In [ ]:
#top 10 bigrams for each class

from nltk import ngrams

for i in range(8):
  filtered_df = df[df['gold_label'] == i]
  flat_data_i = [item for sublist in filtered_df['text'] for item in sublist]
  bigrams = ngrams(flat_data_i, 2)
  top_10_bigrams = Counter(bigrams).most_common(10)
  print(f'Class {i}:')
  print(top_10_bigrams)

In [ ]:
#top 10 trigrams

for i in range(8):
  filtered_df = df[df['gold_label'] == i]
  flat_data_i = [item for sublist in filtered_df['text'] for item in sublist]
  trigrams = ngrams(flat_data_i, 3)
  top_10_trigrams = Counter(trigrams).most_common(10)
  print(f'Class {i}:')
  print(top_10_trigrams)


In [ ]:
text0 = train[train['gold_label']==0]['text'].tolist()
text1 = train[train['gold_label']==1]['text'].tolist()
text2 = train[train['gold_label']==2]['text'].tolist()
text3 = train[train['gold_label']==3]['text'].tolist()
text4 = train[train['gold_label']==4]['text'].tolist()
text5 = train[train['gold_label']==5]['text'].tolist()
text6 = train[train['gold_label']==6]['text'].tolist()
text7 = train[train['gold_label']==7]['text'].tolist()

print('0: ')
print(*text0[:5],sep='\n')
print('1: ')
print(*text1[:5],sep='\n')
print('2: ')
print(*text2[:5],sep='\n')
print('3: ')
print(*text3[:5],sep='\n')
print('4: ')
print(*text4[:5],sep='\n')
print('5: ')
print(*text5[:5],sep='\n')
print('6: ')
print(*text6[:5],sep='\n')
print('7: ')
print(*text7[:5],sep='\n')


In [ ]:
#look at hashtags

import re

def extract_hashtags(text):
    hashtags = re.findall(r'#(\w+)', text)
    return hashtags

text_not7 = df[df['gold_label']!=7]['text'].tolist()
text_not7 = [' '.join(sent) for sent in text_not7]
hashtags_not7 = [extract_hashtags(text) for text in text_not7]
hashtags_7 = [extract_hashtags(text) for text in text7]
#top 20 hashtags:

flat_hashtags_not7 = [item for sublist in hashtags_not7 for item in sublist]
top_20_hashtags = Counter(flat_hashtags_not7).most_common(20)
print(top_20_hashtags)
flat_hashtags_7 = [item for sublist in hashtags_7 for item in sublist]
top_20_hashtags = Counter(flat_hashtags_7).most_common(20)
print(top_20_hashtags)


# *Preprocessing*

## Some trials in preprocessing techniques

In [ ]:
tknzr = TweetTokenizer(strip_handles=True, reduce_len=True)

In [ ]:
tknzr.tokenize('@remy: This is suuuucks too much for you!!!!!!')

In [ ]:
t0 = nltk.tokenize.casual.remove_handles(train.iloc[96, 1])

In [ ]:
t0

In [ ]:
tknzr.tokenize(train.iloc[96, 1])

In [ ]:
t1 = 'Come and lick it up faggots #manscent #pits #muskypits #myripecock #3daystink {URL}'

In [ ]:
tknzr.tokenize(t1)

In [ ]:
word_tokenize(t1)

In [ ]:
parsed = nlp(t1)

[token.text for token in parsed]

In [ ]:
print(seg.segment('muskypits'))

In [ ]:
t0_cleaned = nltk.tokenize.casual.remove_handles(t0)

t0_cleaned

In [ ]:
t2 = "I'll get my revenge when I fuck on your nigga , I'll take that shit back , I won't fuck on your nigga , I'll fuck on his face, right in your place."

In [ ]:
tknzr.tokenize(t2)

In [ ]:
word_tokenize(t2)

In [ ]:
t3 = "@user @user It has come down to this, kill or be killed. The evil ones who planned the flooding of Western nations full of Muslims are the real sons of Satan. Discover who is the mastermind behind this and you know who your real enemy is. It is not the Demonrats, they are a minor player."

In [ ]:
word_tokenize(t3)

In [ ]:
tknzr.tokenize(t3)

In [ ]:
t4 = "@user @user Deadass tho ,, buddy half past retarded for that one 😂😭"

In [ ]:
word_tokenize(t4)

In [ ]:
tknzr.tokenize(t4)

<u>Summary</u>:

* NLTK's regular `word_tokenize` splits @'s into '@' and 'user', while `TweetTokenizer` can get rid of them.

* `word_tokenize` can also split #'s into '#' and the following word. `spacy` can do the same.

* `word_tokenize` does not tokenize emojis, `TweetTokenizer` does.

* `TweetTokenizer` can shorten the length of words ('waaay' --> 'way').

* `TweetTokenizer` is not good at tokenizing contractions or splitting hashtags, it should only be used for @ removal and emoji tokenization.

* `ekphrasis` library can be used to tokenize multi-word hashtags!

____

<u>Suggestion</u>: a pipeline that uses all of these libraries to preprocess tweets:
1. Use `TweetTokenizer` to remove @user and segment emojis
2. Use 'word_tokenize' to tokenize the sentence
3. Use `ekphrasis` to tokenize multi-word hashtags, replace emojis with words
4. Remove hashtags, stopwords, punctuation marks using `spacy` or `nltk`
5. Lemmatize using `spacy` or `nltk`

## Preprocessing functions

In [ ]:
# How many emojis do we have in our data? --> Does it even require handling / remove them?

# GloVe

# Use external lists of hateful words?

In [ ]:
def remove_url(text):
  '''
  This function takes a textual input and uses regex to replace any instance of {URL} with an empty
  string, cleaning ocurrences of {URL} from the data.
  The function then returns the cleaned text where {URL} is present, and returns the original text where {URL} is not present.
  '''
    if '{URL}' in text:
      new_text=re.sub("{URL}", " ", text)
      return new_text
    else:
      return text

In [ ]:
puncts = string.punctuation+'..'+'...'+'•'

In [ ]:
def spell_correction(text):
  '''
  This function takes text data as an input and iterates through line of data using Ekphrasis
  SpellCorrector to correct any spelling mistakes, whilst preserving punctuation and digits.
  It then returns the text with any corrections made.
  '''
  corrected = []
  split_text = text.strip().split()
  for word in split_text:
      if word in puncts:
        corrected.append(word)
      elif word.isdigit():
        corrected.append(word)
      else:
        corrected.append(sp.correct(word))

  if split_text == corrected:
    return text
  else:
    return " ".join(corrected)

In [ ]:
def split_hashtag(text):
  '''
  This function takes text data as an input, and iterates through each line of data to find tokens
  that begin with a '#' to identify hashtags, and then uses Ekphrasis Segmenter so split hashtags
  into separate words.
  The function then returns the text data with multiword hashtags split into monoword tokens.
  '''
    new_t = text.split()

    output = []

    for word in range(len(new_t)):
        if new_t[word-1] == '#':
            # new_w = word.split('#')[1]
            output.append(seg.segment(new_t[word]))

        else:
            output.append(new_t[word])

    return " ".join(output)

In [ ]:
import re

def count_hashtags(text):
    hashtag_pattern = re.compile(r"#\w+")
    return len(hashtag_pattern.findall(text))

In [ ]:
def remove_emojis(string):
    # Unicode ranges for emojis
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Miscellaneous symbols and pictographs
        "\U0001F680-\U0001F6FF"  # Transport and map symbols
        "\U0001F1E0-\U0001F1FF"  # Flags (iOS)
        "\U00002700-\U000027BF"  # Dingbats
        "\U00002600-\U000026FF"  # Miscellaneous symbols
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        "\U0001F018-\U0001F0FF"  # Playing cards
        "\U00002B50-\U00002B55"  # Stars and other symbols
        "\U00002300-\U000023FF"  # Miscellaneous technical
        "]+", flags=re.UNICODE)

    return emoji_pattern.sub(r'', string)

In [ ]:
def contains_emoji(string):
    # Unicode ranges for emojis
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Miscellaneous symbols and pictographs
        "\U0001F680-\U0001F6FF"  # Transport and map symbols
        "\U0001F1E0-\U0001F1FF"  # Flags (iOS)
        "\U00002700-\U000027BF"  # Dingbats
        "\U00002600-\U000026FF"  # Miscellaneous symbols
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        "\U0001F018-\U0001F0FF"  # Playing cards
        "\U00002B50-\U00002B55"  # Stars and other symbols
        "\U00002300-\U000023FF"  # Miscellaneous technical
        "]+", flags=re.UNICODE)

    return bool(emoji_pattern.search(string))

In [ ]:
def count_emojis(text):
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Miscellaneous symbols and pictographs
        "\U0001F680-\U0001F6FF"  # Transport and map symbols
        "\U0001F1E0-\U0001F1FF"  # Flags (iOS)
        "\U00002700-\U000027BF"  # Dingbats
        "\U00002600-\U000026FF"  # Miscellaneous symbols
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        "\U0001F018-\U0001F0FF"  # Playing cards
        "\U00002B50-\U00002B55"  # Stars and other symbols
        "\U00002300-\U000023FF"  # Miscellaneous technical
        "]+", flags=re.UNICODE)

    return len(emoji_pattern.findall(text))

In [ ]:
def get_tokenized_lemmas(text):

  doc = nlp(f"{text}")

  return " ".join([token.lemma_ for token in doc])

In [ ]:
emo = Translator(exact_match_only=False,
                 exclude_stopwords=True,
                 randomize=True)

In [ ]:
emo.get_word('😡')

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text, stopwords=True):
  '''
  This function takes text data as an input and takes stopwords as true.
  It compiles a range of NLTK modules, with python functions and user-defined functions to clean the
  text data as follows: Username handles are removed, instances of {URL} are removed, the text is tokenized
  and lemmatized, multi-word hashtags are segmented, punctuation marks are removed, the text is put
  in lowercase, lengthening is reduced, emojis are removed, spelling mistakes are corrected and stopwords are removed.
  The function then returns the cleaned text
  '''

    # REPLACE_BY_SPACE_RE = re.compile('[/(){}\[\]\|@,;]')
    # text = REPLACE_BY_SPACE_RE.sub(' ', text) # replace REPLACE_BY_SPACE_RE symbols by space in text. substitute the matched string in REPLACE_BY_SPACE_RE with space.
    # text = BAD_SYMBOLS_RE.sub('', text) # remove symbols which are in BAD_SYMBOLS_RE from text. substitute the matched string in BAD_SYMBOLS_RE with nothing.
    # text = text.replace('x', '')
    # text = re.sub(r'\W+', '', text)

    # remove handles
    text = nltk.tokenize.casual.remove_handles(text)

    # remove url
    text = remove_url(text)

    # tokenize and lemmatize
    text = get_tokenized_lemmas(text)

    # split multi-word hashtags
    text = split_hashtag(text)

    # remove punctuation marks
    text = text.translate(str.maketrans('', '', puncts))

    # lowercase text
    text = text.lower()

    # reduce lengthening
    text = nltk.tokenize.casual.reduce_lengthening(text)

    # delete emojis
    text = remove_emojis(text)

    # correct spelling
    text = spell_correction(text)

    # remove stopwords from text
    if stopwords:
      text = ' '.join(word for word in text.split() if word not in stop_words)

    # tokenization: handle emojis?

    return text

In [ ]:
def count_punctuation(text):
    text = nltk.tokenize.casual.remove_handles(text)

    text = remove_url(text)

    text = split_hashtag(text)

    return sum(1 for char in text if char in puncts)

In [ ]:
# Corpus-specific stop word list?

# *Data preparation*

## Data sets with stop words

In [ ]:
train_data_sw = deepcopy(train)

train_data_sw['cleaned'] = train_data_sw['text'].apply(clean_text, stopwords = False)

validation_data_sw = deepcopy(validation)

validation_data_sw['cleaned'] = validation_data_sw['text'].apply(clean_text, stopwords = False)

test_data_sw = deepcopy(test)

test_data_sw['cleaned'] = test_data_sw['text'].apply(clean_text, stopwords = False)

In [ ]:
# test_data_sw.to_csv('preprocessed training data without stop words.csv')

# test_data_sw.to_csv('preprocessed validation data without stop words.csv')

# test_data_sw.to_csv('preprocessed test data without stop words.csv')

# !cp 'preprocessed training data without stop words.csv' "/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/"
# !cp 'preprocessed validation data without stop words.csv' "/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/"
# !cp 'preprocessed test data without stop words.csv' "/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/"

## Data sets without stop words

In [ ]:
train_data = deepcopy(train)

train_data['cleaned'] = train_data['text'].apply(clean_text)

train_data['contains emojis'] = train_data['text'].apply(contains_emoji)

train_data['punctuation count'] = train_data['text'].apply(count_punctuation)

validation_data = deepcopy(validation)

validation_data['cleaned'] = validation_data['text'].apply(clean_text)

validation_data['punctuation count'] = validation_data['text'].apply(count_punctuation)

validation_data['string length'] = validation_data['text'].apply(lambda words: len(words.split(" ")))

validation_data['contains emojis'] = validation_data['text'].apply(contains_emoji)

test_data = deepcopy(test)

test_data['cleaned'] = test_data['text'].apply(clean_text)

test_data['punctuation count'] = test_data['text'].apply(count_punctuation)

test_data['string length'] = test_data['text'].apply(lambda words: len(words.split(" ")))

test_data['contains emojis'] = test_data['text'].apply(contains_emoji)

In [ ]:
train_data['contains emojis'].replace(to_replace = {False: 0, True: 1}, inplace=True)

validation_data['contains emojis'].replace(to_replace = {False: 0, True: 1}, inplace=True)

test_data['contains emojis'].replace(to_replace = {False: 0, True: 1}, inplace=True)

In [ ]:
# train_data.rename(columns={'tweet length':'string length'}, inplace=True)

# validation_data.rename(columns={'tweet length':'string length'}, inplace=True)

# test_data.rename(columns={'tweet length':'string length'}, inplace=True)

In [ ]:
# train_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/preprocessed training data.csv', index_col=0)

# validation_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/preprocessed validation data.csv', index_col=0)

# test_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CMT122 Machine Learning/Coursework 2/preprocessed testing data.csv', index_col=0)

In [ ]:
train_data['emoji count'] = train_data['text'].apply(count_emojis)

validation_data['emoji count'] = validation_data['text'].apply(count_emojis)

test_data['emoji count'] = test_data['text'].apply(count_emojis)

train_data['hashtag count'] = train_data['text'].apply(count_hashtags)

validation_data['hashtag count'] = validation_data['text'].apply(count_hashtags)

test_data['hashtag count'] = test_data['text'].apply(count_hashtags)

## Feature selection using ANOVA

In [ ]:
from sklearn.feature_selection import f_classif

features = ['string length', 'contains emojis', 'punctuation count',
            'hashtag count', 'emoji count']

X_train_num = train_data[features]

F_values, p_values = f_classif(X_train_num, y)

anova_results = pd.DataFrame({
    'Feature': X_train_num.columns,
    'F-Statistic': F_values,
    'P-Value': p_values
})
print(anova_results)

In [ ]:
X_train_num_selected = X_train_num[selected_features]
X_val_num_selected = X_val_num[selected_features]

# Combine selected numeric features with TF-IDF features
X_train_combined = np.hstack((cleaned_tfidf, X_train_num_selected))
X_val_combined = np.hstack((cleaned_tfidf_val, X_val_num_selected))

# Train logistic regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_combined, y)

# Evaluate on validation data
y_pred_combined = log_reg.predict(X_val_combined)
combined_accuracy = accuracy_score(y_val, y_pred_combined)

combined_accuracy

In [ ]:
print(classification_report(y_val, y_pred_combined))

## Vectorization

In [ ]:
def identity_tokenizer(text):
    return text

In [ ]:
# # Prepare training data for fitting

# tfidf = TfidfVectorizer(max_features=2000,   # Limit to 500 most important words
#                         ngram_range=(1, 4))

# cleaned_tfidf = tfidf.fit_transform(train_data['cleaned']).toarray()

# features = ['string length', 'contains emojis', 'punctuation count',
#             'hashtag count', 'emoji count']

# other_features = train_data[features].values

# X = pd.concat([pd.DataFrame(cleaned_tfidf), pd.DataFrame(other_features)], axis=1)

# y = train_data['gold_label']

# # Prepare validation data for prediction

# cleaned_tfidf_val = tfidf.transform(validation_data['cleaned']).toarray()
# other_features_val = validation_data[features].values
# X_val_2 = pd.concat([pd.DataFrame(cleaned_tfidf_val), pd.DataFrame(other_features_val)], axis=1)
# y_val = validation_data['gold_label']

# # Prepare test data for prediction

# cleaned_tfidf_test = tfidf.transform(test_data['cleaned']).toarray()
# other_features_test = test_data[features].values
# X_test_2 = pd.concat([pd.DataFrame(cleaned_tfidf_test), pd.DataFrame(other_features_test)], axis=1)
# y_test = test_data['gold_label']

### With stop words

In [ ]:
features = ['string length','punctuation count']

In [ ]:
train_data_sw['punctuation count'] = train_data_sw['text'].apply(count_punctuation)

train_data_sw['string length'] = train_data_sw['text'].apply(lambda words: len(words.split(" ")))

validation_data_sw['punctuation count'] = validation_data_sw['text'].apply(count_punctuation)

validation_data_sw['string length'] = validation_data_sw['text'].apply(lambda words: len(words.split(" ")))

test_data_sw['punctuation count'] = test_data_sw['text'].apply(count_punctuation)

test_data_sw['string length'] = test_data_sw['text'].apply(lambda words: len(words.split(" ")))

In [ ]:
tfidf_sw = TfidfVectorizer(max_features=2000,
                        ngram_range=(1, 4),
                        tokenizer=identity_tokenizer)

cleaned_tfidf_sw = tfidf_sw.fit_transform(train_data_sw['cleaned']).toarray()

# Training

other_features_sw = train_data_sw[features].values
X_sw = pd.concat([pd.DataFrame(cleaned_tfidf_sw), pd.DataFrame(other_features_sw)],
              axis=1)

y_sw = train_data_sw['gold_label']

# Validation
cleaned_tfidf_val_sw = tfidf_sw.transform(validation_data_sw['cleaned']).toarray()

other_features_val_sw = validation_data_sw[features].values

X_val_sw = pd.concat([pd.DataFrame(cleaned_tfidf_val_sw),
                     pd.DataFrame(other_features_val_sw)],
                    axis=1)

y_val_sw = validation_data_sw['gold_label']

# Test
cleaned_tfidf_test_sw = tfidf_sw.transform(test_data_sw['cleaned']).toarray()

other_features_test_sw = test_data_sw[features].values

X_test_sw = pd.concat([pd.DataFrame(cleaned_tfidf_test_sw),
                      pd.DataFrame(other_features_test_sw)],
                     axis=1)

y_test_sw = test_data_sw['gold_label']

### Without stop words

In [ ]:
train_data['punctuation count'] = train_data['text'].apply(count_punctuation)

train_data['string length'] = train_data['text'].apply(lambda words: len(words.split(" ")))

validation_data['punctuation count'] = validation_data['text'].apply(count_punctuation)

validation_data['string length'] = validation_data['text'].apply(lambda words: len(words.split(" ")))

test_data['punctuation count'] = test_data['text'].apply(count_punctuation)

test_data['string length'] = test_data['text'].apply(lambda words: len(words.split(" ")))

In [ ]:
tfidf = TfidfVectorizer(max_features=2000,
                        ngram_range=(1, 4),
                        tokenizer=identity_tokenizer)

cleaned_tfidf = tfidf.fit_transform(train_data['cleaned']).toarray()

features = ['string length','punctuation count']

# Training

other_features = train_data[features].values
X = pd.concat([pd.DataFrame(cleaned_tfidf), pd.DataFrame(other_features)], axis=1)

y = train_data['gold_label']

# Validation
cleaned_tfidf_val = tfidf.transform(validation_data['cleaned']).toarray()

other_features_val = validation_data[features].values

X_val = pd.concat([pd.DataFrame(cleaned_tfidf_val),
                     pd.DataFrame(other_features_val)],
                    axis=1)

y_val = validation_data['gold_label']

# Test
cleaned_tfidf_test = tfidf.transform(test_data['cleaned']).toarray()

other_features_test = test_data[features].values

X_test = pd.concat([pd.DataFrame(cleaned_tfidf_test),
                      pd.DataFrame(other_features_test)],
                     axis=1)

y_test = test_data['gold_label']

In [ ]:
# Features:
# 1) 'contains user'
# 2) 'number of emojis'
# 3) Capitalization?
# 4) 'contains hashtags' --> value_counts
# 5) Translate emojis?

# Feature selection: add a statistical component (R^2)

# Model:
# 1) Regularization
# 2) Include n-grams?

# *Modeling Logistic Regression*

## With stop words

In [ ]:
lr_model_base = LogisticRegression(max_iter=1000)

lr_model_base.fit(cleaned_tfidf_sw, y_sw)

# Base model, validation

y_pred_base_sw = lr_model_base.predict(cleaned_tfidf_val_sw)

print(classification_report(y_val_sw, y_pred_base_sw))

In [ ]:
lr_with_feats = LogisticRegression(max_iter=1000)

lr_with_feats.fit(X_sw, y_sw)

# Model with features, validation
y_pred_feats_sw = lr_with_feats.predict(X_val_sw)

print(classification_report(y_val_sw, y_pred_feats_sw))

In [ ]:
# Base model, test
y_pred_test_base_sw = lr_model_base.predict(cleaned_tfidf_test_sw)

print(classification_report(y_test_sw, y_pred_test_base_sw))

In [ ]:
# Model with features, test

y_pred_test_feats_sw = lr_with_feats.predict(X_test_sw)

print(classification_report(y_test_sw, y_pred_test_feats_sw))

## Without stop words

In [ ]:
lr_model_base = LogisticRegression(max_iter=1000)

lr_model_base.fit(cleaned_tfidf, y)

# Base model, validation

y_pred_base_val = lr_model_base.predict(cleaned_tfidf_val)

print(classification_report(y_val, y_pred_base_val))

In [ ]:
lr_with_feats = LogisticRegression(max_iter=1000)

lr_with_feats.fit(X, y)

# Model with features, validation
y_pred_feats_val = lr_with_feats.predict(X_val)

print(classification_report(y_val, y_pred_feats_val))

In [ ]:
# Base model, test
y_pred_base_test = lr_model_base.predict(cleaned_tfidf_test)

print(classification_report(y_test, y_pred_base_test))

In [ ]:
# Model with features, test

y_pred_feats_test = lr_with_feats.predict(X_test)

print(classification_report(y_test, y_pred_feats_test))

# Error analysis - Ian

In [ ]:
# #hatecheck = pd.read_csv('/content/drive/My Drive/CMT122 Machine Learning/Coursework 2/test_suite_cases.csv')

# from datasets import load_dataset
# hatecheck = load_dataset("Paul/hatecheck")

Extracting confusion matrix values and mapping them out.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_test2)
# print(cm)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'{labels[i]}' for i in range(8)],
            yticklabels=[f'{labels[i]}' for i in range(8)])

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix of the Test Results - Model with Features')
plt.show()

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
# print(cm)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'{labels[i]}' for i in range(8)],
            yticklabels=[f'{labels[i]}' for i in range(8)])

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix of the Test Results - Model without Features')
plt.show()

Putting them into TP, TN, FP, FN.

In [ ]:
num_classes = cm.shape[0]

for num in range(num_classes):
    TP = cm[num, num]
    FN = np.sum(cm[num, :]) - TP
    FP = np.sum(cm[:, num]) - TP
    TN = np.sum(cm) - (TP + FN + FP)

print(f"  TP: {TP}")
print(f"  FN: {FN}")
print(f"  FP: {FP}")
print(f"  TN: {TN}")



In [ ]:
from collections import defaultdict

misclassified_indices = np.where(y_pred_test2 != y_test)[0]

misclassified_by_class = defaultdict(list)

for idx in misclassified_indices:
    true_class = y_test[idx]
    predicted_class = y_pred_test2[idx]
    misclassified_by_class[(true_class, predicted_class)].append(idx)


misclassified_samples = {}
for (true_class, predicted_class), indices in misclassified_by_class.items():
    misclassified_samples[(true_class, predicted_class)] = {
        "samples": train_data.iloc[indices],
        "indices": indices
    }

print("Misclassified Samples by Class:")
for (true_class, predicted_class), details in misclassified_samples.items():
    print(f"True Class: {true_class}, Predicted Class: {predicted_class}")
    print(f"Samples: {details['samples']}")
    print(f"Indices: {details['indices']}")
    print("-" * 40)

In [ ]:
# Print-out looks a bit rough, so will make an excel file to observe data a little easier.
import pandas as pd
from copy import deepcopy
from google.colab import files

rows = []
for (true_class, predicted_class), indices in misclassified_by_class.items():
    for idx in indices:
        if isinstance(test_data, list):
            sample = test_data[idx]["text"]
        elif isinstance(test_data, pd.DataFrame):
            sample = test_data.iloc[idx]["text"]
        elif isinstance(test_data, np.ndarray):
            sample = test_data[idx, 1]
        else:
            raise ValueError("Unsupported train_data type")

        rows.append({
            "True Class": true_class,
            "Predicted Class": predicted_class,
            "Sample": sample,
            "Index": idx
        })


df2 = pd.DataFrame(rows)


df2 = df2.sort_values(by=["True Class", "Predicted Class", "Index"], ascending=[True, True, True])


excel_filename = "misclassified_samples_sorted.xlsx"
df2.to_excel(excel_filename, index=False)


files.download(excel_filename)





# *Other attempts*

## Classifier 1: LSTM

In [ ]:
# 1) Make a corpus-sensitive list of stop words
    # Disabled stop word removal for now

# 2) Try a different word embedding technique
# - TfidfVectorizer

# 3) Check techniques, papers, articles for class imbalances
# - Undersampling class 7 / Oversampling classes 0-6
# - Hierarchical samples: class 7 against 0-6, then 0-6 against each other


# Consider another model:
# - A simple naive Bayes model on top of bag of words.


# Code examples:

# https://stackoverflow.com/questions/77455103/low-recall-and-f1-score-for-lstm-text-classification

# https://datascience.stackexchange.com/questions/13490/how-to-set-class-weights-for-imbalanced-classes-in-keras

# https://www.digitalocean.com/community/tutorials/tensorflow-callbacks

# https://datascience.stackexchange.com/questions/117456/lstm-model-is-producing-really-bad-results-for-multiclass-text-classification-fo

# https://datascience.stackexchange.com/questions/93074/how-to-improve-lstm-accuracy-on-multiclass-text-classification

# https://datascience.stackexchange.com/questions/27888/imbalanced-data-causing-mis-classification-on-multiclass-dataset

### *FastText*

In [ ]:
texts = train_data['cleaned'].tolist()
labels = train_data['gold_label'].tolist()

tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_seq_len, padding='post')
word_index = tokenizer.word_index

In [ ]:
# FastText:
import gensim
from gensim.models import FastText

# fasttext_model = gensim.models.KeyedVectors.load_word2vec_format(
#     'crawl-300d-2M.vec', binary=False
# )

# # Create the embedding matrix
# embedding_dim = 300
# vocab_size = len(word_index) + 1
# embedding_matrix = np.zeros((vocab_size, embedding_dim))

# for word, i in word_index.items():
#     if word in fasttext_model:
#         embedding_matrix[i] = fasttext_model[word]

lemmatized_cells = list(train_data['cleaned'].apply(str.split))

vector_size = 300

ft_model = FastText(vector_size = vector_size, window = 5, min_count = 3)
ft_model.build_vocab(corpus_iterable = lemmatized_cells)
ft_model.train(corpus_iterable = lemmatized_cells,
               total_examples = len(lemmatized_cells),
               epochs=4)

In [ ]:
word_vectors = ft_model.wv

embedding_matrix = np.zeros((vocab_size+1, vector_size))

for word, i in word_index.items():
    if word in word_vectors:
        embedding_matrix[i] = word_vectors[word]

In [ ]:
# ft_model.save("custom_fasttext.model")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Define the model
model = Sequential([
    Embedding(
        input_dim=vocab_size+1,
        output_dim=vector_size,
        weights=[embedding_matrix],
        # input_length=max_seq_len,
        trainable=False
    ),
    LSTM(128, return_sequences=False),
    Dense(8, activation='softmax')
])

model.add(Dropout(0.5))

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
# # Train the model
# model.fit(
#     np.array(padded_sequences),
#     np.array(labels),
#     epochs=12,
#     batch_size=32
# )

In [ ]:
# test_accuracy = model.evaluate(np.array(padded_sequences), np.array(labels))
# print(f"Test Accuracy: {test_accuracy[1] * 100:.2f}%")

Validation dataset

In [ ]:
val_texts = list(validation_data['cleaned'])

val_labels = list(validation_data['gold_label'])

val_sequences = tokenizer.texts_to_sequences(val_texts)

val_padded = pad_sequences(val_sequences, maxlen=max_seq_len, padding='post')

callback = tensorflow.keras.callbacks.EarlyStopping(monitor='loss', patience=3, min_delta=0.0001)

model.fit(
    np.array(padded_sequences),
    np.array(labels),
    validation_data=(np.array(val_padded),
                     np.array(val_labels)),
    validation_split=0.1,
    callbacks = [callback],
    epochs=10,
    batch_size=32
)

In [ ]:
# Plot accuracy and val_accuracy vs. num of epochs

In [ ]:
test_texts = list(test_data['cleaned'])

test_labels = list(test_data['gold_label'])

test_sequences = tokenizer.texts_to_sequences(test_texts)

test_padded = pad_sequences(test_sequences, maxlen=max_seq_len, padding='post')

test_loss, test_accuracy = model.evaluate(np.array(test_padded), np.array(test_labels))

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
predictions = model.predict(np.array(test_padded))

predicted_classes = predictions.argmax(axis=1)

# for i, (predicted, actual) in enumerate(zip(predicted_classes, test_labels)):
#     print(f"Example {i}: Predicted = {predicted}, Actual = {actual}")

# Generate confusion matrix
cm = confusion_matrix(test_labels, predicted_classes)
print("Confusion Matrix:\n", cm)

# Detailed classification report
report = classification_report(test_labels, predicted_classes)
print("Classification Report:\n", report)

In [ ]:
# # Adjusting the weights to include

# from sklearn.utils.class_weight import compute_class_weight
# import numpy as np

# # Compute class weights
# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(labels),
#     y=labels
# )
# class_weights = {i: weight for i, weight in enumerate(class_weights)}

# # Pass class weights to model.fit()
# model.fit(
#     np.array(padded_sequences),
#     np.array(labels),
#     validation_data=(np.array(val_padded), np.array(val_labels)),
#     epochs=12,
#     batch_size=32,
#     class_weight=class_weights
# )

In [ ]:
test_loss, test_accuracy = model.evaluate(np.array(test_padded), np.array(test_labels), verbose=1)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
predictions = model.predict(np.array(test_padded))

predicted_classes = predictions.argmax(axis=1)

cm = confusion_matrix(test_labels, predicted_classes)
print("Confusion Matrix:\n", cm)

# Detailed classification report
report = classification_report(test_labels, predicted_classes)
print("Classification Report:\n", report)

### *TF-IDF*

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class LinguisticFeaturesExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ensure numerical columns are converted to numpy arrays
        return np.array(X[self.columns])

In [ ]:
def identity_tokenizer(text):
    return text

In [ ]:
vectorizer = TfidfVectorizer(tokenizer=identity_tokenizer,
                             ngram_range = (1, 4),
                              max_df = 0.8
                            #  , max_features=1000
                              )

In [ ]:
linguistic_features = ['string length', 'contains emojis', 'punctuation count']

preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(tokenizer=identity_tokenizer,
                             ngram_range = (1, 4),
                              max_df = 0.8), 'cleaned'),
        ('linguistic',
         'passthrough'
        #  Pipeline([
        #     ('extract', LinguisticFeaturesExtractor(linguistic_features)),
        #     ('scale', StandardScaler())
        # ])
         , linguistic_features)
        ],
    remainder='drop')

In [ ]:
class_7 = train_data[train_data['gold_label'] == 7].sample(n = 300, random_state = 42)
class_0 = train_data[train_data['gold_label'] == 0].sample(n = 300, random_state = 42)
other_classes = train_data[train_data['gold_label'].isin([1,2,3,4,5,6])]

train_data = pd.concat([class_7, class_0, other_classes])

In [ ]:
# Fit the preprocessor to training data
X_train_preprocessed = preprocessor.fit_transform(train_data)
X_val_preprocessed = preprocessor.transform(validation_data)

# Pad the TF-IDF vectors (first part of X_train_preprocessed)
tfidf_train = pad_sequences(X_train_preprocessed[:, 0].toarray(),
                             padding='post', maxlen=1000)
tfidf_val = pad_sequences(X_val_preprocessed[:, 0].toarray(),
                           padding='post', maxlen=1000)

# Concatenate linguistic features with padded TF-IDF vectors
linguistic_train = X_train_preprocessed[:, 1].toarray()
linguistic_val = X_val_preprocessed[:, 1].toarray()

X_train_combined = np.hstack([tfidf_train, linguistic_train])
X_val_combined = np.hstack([tfidf_val, linguistic_val])

y_train = np.array(train_data['gold_label'])
y_val = np.array(validation_data['gold_label'])

In [ ]:
# X_train_vectors = preprocessor.fit_transform(train_data)
# y = np.array(train_data['gold_label'])

# X_val_vectors = preprocessor.transform(validation_data)
# y_val = np.array(list(validation_data['gold_label']))

# X_train_padded = pad_sequences(X_train_vectors.toarray(),
#                                padding='post',
#                                maxlen=1000)

# X_val_padded = pad_sequences(X_val_vectors.toarray(),
#                              padding='post',
#                              maxlen=1000)

In [ ]:
model1 = Sequential([
    Input(shape=(X_train_combined.shape[1],)),
    Embedding(input_dim=1000, output_dim=128, input_length=X_train_combined.shape[1]),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dense(128, activation='relu'),
    Dense(8, activation='softmax')
])

model1.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model1.fit(x = X_train_combined,
           y = y_train,
           epochs=20,
           batch_size=100,
           shuffle=True,
            # validation_data=(X_val_combined, y_val),
           verbose=1)

In [ ]:
prediction = model1.predict(X_val_combined)

emotion_pred = np.argmax(prediction, axis = 1)
emotion_pred

In [ ]:
loss1, accuracy1 = model1.evaluate(X_val_combined, y_val)
print(f"Validation Loss: {loss1:.4f}, Validation Accuracy: {accuracy1:.4f}")

In [ ]:
X_test_preprocessed = preprocessor.transform(test_data)

tfidf_test = pad_sequences(X_test_preprocessed[:, 0].toarray(),
                           padding='post', maxlen=1000)

linguistic_test = X_test_preprocessed[:, 1].toarray()

X_test_combined = np.hstack([tfidf_test, linguistic_test])

y_test = np.array(test_data['gold_label'])

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# X_test_vectors = vectorizer.transform(list(test_data['cleaned']))
# X_test_padded = pad_sequences(X_test_vectors.toarray(), padding='post', maxlen=1000)
# y_test = np.array(test_data['gold_label'])

# Step 2: Evaluate the model on the test dataset
test_loss1, test_accuracy1 = model1.evaluate(X_test_combined, y_test)
print(f"Test Loss: {test_loss1:.4f}, Test Accuracy: {test_accuracy1:.4f}")

In [ ]:
# Step 3: Predict labels for the test dataset
y_pred = model1.predict(X_test_combined)
y_pred_classes = np.argmax(y_pred, axis=1)

# Step 4: Generate confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)
print("Confusion Matrix:")
print(cm)

In [ ]:
# # Visualize the confusion matrix
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=range(8), yticklabels=range(8))
# plt.xlabel('Predicted Labels')
# plt.ylabel('True Labels')
# plt.title('Confusion Matrix')
# plt.show()

In [ ]:
# Step 5: Classification report
print("Classification Report:")
print(classification_report(y_test,
                            y_pred_classes
                            # ,                           target_names=[str(i) for i in range(8)]
                            ))

Introduce class weights:

In [ ]:
class_weights = class_weight.compute_class_weight(class_weight = 'balanced',
                                                 classes = np.unique(y),
                                                 y = y)

class_weights = dict(zip(np.unique(y), y))

In [ ]:
model = Sequential([
    Input(shape=(X_train_padded.shape[1],)),
    Embedding(input_dim=1000, output_dim=128, input_length=X_train_padded.shape[1]),  # 128 is the embedding dimension
    Bidirectional(LSTM(64, return_sequences=False)),
    Dense(128, activation='relu'),
    Dense(8, activation='softmax')  # 8 categories for classification
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train_padded, y,
                    class_weight=class_weights,
                    epochs=10,  # Adjust the number of epochs as needed
                    batch_size=64,
                    # validation_data=(X_val_padded, y_val),
                    verbose=1)

In [ ]:
X_test_vectors = vectorizer.transform(list(test_data['cleaned']))
X_test_padded = pad_sequences(X_test_vectors.toarray(), padding='post', maxlen=1000)
y_test = np.array(test_data['gold_label'])

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_padded, y_test)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

In [ ]:
y_pred = model.predict(X_test_padded)
y_pred_classes = np.argmax(y_pred, axis=1)

# Step 4: Generate confusion matrix
confusion_matrix(y_test, y_pred_classes)

In [ ]:
# Step 5: Classification report
print("Classification Report:")
print(classification_report(y_test,
                            y_pred_classes
                            # ,                           target_names=[str(i) for i in range(8)]
                            ))

Over-sampling:

In [ ]:
# did not work
# from https://towardsdatascience.com/classification-framework-for-imbalanced-data-9a7961354033

def over_sample(data):
  # data = scaler(data)
  to_oversample = data[data['gold_label'].isin([0,1,2,3,4,5,6])]
  class_7 = data[data['gold_label'] == 7]

  upsampled_data = pd.DataFrame.resample(to_oversample, replacement=True, # sample with replacement
                                         n_samples = len(class_7), # match number in majority class
                                         random_state = 42) # reproducible results

  data = pd.concat([upsampled_data, class_7])
  return data

resampled_data = over_sample(train_data)

In [ ]:
max_size = train_data['gold_label'].value_counts().max()

lst = []

for class_index, group in train_data.groupby('gold_label'):
    lst.append(group.sample(500, replace=True))

resampled_data = pd.concat(lst)

In [ ]:
resampled_data['gold_label'].value_counts()

In [ ]:
vectorizer2 = TfidfVectorizer(tokenizer=identity_tokenizer,
                             ngram_range = (1, 4),
                              max_df = 0.7,
                              max_features=1000
                              )

X_train_vectors2 = vectorizer2.fit_transform(list(resampled_data['cleaned']))
y2 = np.array(resampled_data['gold_label'])

X_val_vectors = vectorizer2.transform(list(validation_data['cleaned']))
y_val = np.array(list(validation_data['gold_label']))

X_train_padded2 = pad_sequences(X_train_vectors2.toarray(),
                               padding='post',
                               maxlen=1000)

X_val_padded = pad_sequences(X_val_vectors.toarray(),
                             padding='post',
                             maxlen=1000)

In [ ]:
model2 = Sequential([
    Input(shape=(X_train_padded2.shape[1],)),
    Embedding(input_dim=1000, output_dim=128, input_length=X_train_padded2.shape[1]),  # 128 is the embedding dimension
    Bidirectional(LSTM(64, return_sequences=False)),
    Dense(128, activation='relu'),
    Dense(8, activation='softmax')
])

model2.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history2 = model2.fit(X_train_padded2, y2,
                    epochs=3,
                    batch_size=64,
                    validation_data=(X_val_padded, y_val),
                    verbose=1)

## Classifier 2: Random Forest

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# data = pd.read_csv('/content/preprocessed training data.csv', index_col=0)

# val_data_2 = pd.read_csv('/content/preprocessed validation data.csv', index_col=0)

# test_data_2 = pd.read_csv('/content/preprocessed test data.csv', index_col=0)

In [ ]:
data = deepcopy(train_data)

val_data_2 = deepcopy(validation_data)

test_data_2 = deepcopy(test_data)

In [ ]:
# Random forest model without features

rf_model = RandomForestClassifier(n_estimators=100,
                                  class_weight='balanced',
                                  random_state=42)

rf_model.fit(cleaned_tfidf, y)

In [ ]:
# Prediction using validation data, without features

y_pred_rf_no_feats = rf_model.predict(cleaned_tfidf_val)

print(classification_report(y_val, y_pred_rf_no_feats))

In [ ]:
# RF model with features
rf_model_with_feats = RandomForestClassifier(n_estimators=100,
                                           class_weight='balanced',
                                           random_state=42)

rf_model_with_feats.fit(X, y)

In [ ]:
# Prediction using validation data, with features

y_pred_feats = rf_model.predict(X_val_2)

print(classification_report(y_val, y_pred_feats))

In [ ]:
# Base, no features

y_pred_test_no_feats = rf_model.predict(cleaned_tfidf_test)

print(classification_report(y_test, y_pred_test_no_feats))

In [ ]:
# With features

y_pred_test_feats = rf_model_with_feats.predict(X_test_2)

print(classification_report(y_test, y_pred_test_feats))